<a href="https://colab.research.google.com/github/isaacadebayo/Predictive-Analytics-Public-Datasets/blob/main/Image_Generation_using_gpt_image_1.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [1]:
import gradio as gr
from openai import OpenAI
import os


In [2]:
# Upgrade the openai library to ensure compatibility with the latest models
!pip install --upgrade openai

# Using Open AI GPT 4 model

In [12]:
from google.colab import userdata
# Change 'secretName' to the actual name of your secret, e.g., 'OPENAI_API_KEY'

In [13]:
# To fix this, set your OpenAI API key. For example, you can uncomment the line below
# os.environ["OPENAI_API_KEY"] = "YOUR_OPENAI_API_KEY"
# Or, use Colab's built-in secrets management (recommended for security):
from google.colab import userdata
os.environ["OPENAI_API_KEY"] = userdata.get("OPENAI_API_KEY")

client = OpenAI(api_key=os.environ.get("OPENAI_API_KEY"))

### Using Colab's Secrets Management for API Keys

1.  **Open the Secrets Panel:** On the left sidebar of your Colab notebook, click the "🔑 Secrets" icon.
2.  **Add a New Secret:** Click on "+ New secret".
3.  **Name the Secret:** In the "Name" field, type `OPENAI_API_KEY` (it's important to use this exact name as your code expects it).
4.  **Enter Your API Key:** In the "Value" field, paste your actual OpenAI API key.
5.  **Save:** Click "Save secret".
6.  **Enable Notebook Access:** Make sure the "Notebook access" toggle is turned on for this secret, so your notebook can access it.

Now, your code can securely retrieve this key without hardcoding it directly into the notebook.

# Text to Image Prompt using Gradio

In [27]:
import base64
from io import BytesIO
from PIL import Image

def text_to_image(prompt):
  if not prompt:
    return None

  r = client.images.generate(
  model="gpt-image-1",
  prompt=prompt,
  size="1024x1024",
  )

  b64 = r.data[0].b64_json
  img_bytes = base64.b64decode(b64)
  return Image.open(BytesIO(img_bytes))

# Stage 1 UI
with gr.Blocks() as stage_1:
    gr.Markdown("# Stage 1: Text to Vegetable Image")
    text_input = gr.Textbox(label="Describe your vegetable...")
    image_output = gr.Image(label="Generated Result")
    submit_btn = gr.Button("Generate")

    submit_btn.click(fn=text_to_image, inputs=text_input, outputs=image_output)

stage_1.launch(debug=True)

It looks like you are running Gradio on a hosted Jupyter notebook, which requires `share=True`. Automatically setting `share=True` (you can turn this off by setting `share=False` in `launch()` explicitly).

Colab notebook detected. This cell will run indefinitely so that you can see errors and logs. To turn off, set debug=False in launch().
* Running on public URL: https://60118da490389c0f31.gradio.live

This share link expires in 1 week. For free permanent hosting and GPU upgrades, run `gradio deploy` from the terminal in the working directory to deploy to Hugging Face Spaces (https://huggingface.co/spaces)


Keyboard interruption in main thread... closing server.
Killing tunnel 127.0.0.1:7860 <> https://60118da490389c0f31.gradio.live


In [17]:
! pip show openai

Name: openai
Version: 2.38.0
Summary: The official Python library for the openai API
Home-page: https://github.com/openai/openai-python
Author: 
Author-email: OpenAI <support@openai.com>
License: Apache-2.0
Location: /usr/local/lib/python3.12/dist-packages
Requires: anyio, distro, httpx, jiter, pydantic, sniffio, tqdm, typing-extensions
Required-by: 


# Audio to image prompt using Gradio

In [30]:
def audio_to_image(audio_path):
    if audio_path is None:
        return None, "No audio recorded"

    # 1. Transcribe the spoken audio using Whisper
    with open(audio_path, "rb") as audio_file:
        transcript = client.audio.transcriptions.create(
            model="whisper-1",
            file=audio_file
        )

    spoken_text = transcript.text

    # 2. Re-use the generation logic from Stage 1
    image_url = text_to_image(spoken_text)

    return image_url, spoken_text

# Stage 2 UI
with gr.Blocks() as stage_2:
    gr.Markdown("# Stage 2: Voice to Vegetable Image")
    audio_input = gr.Audio(type="filepath", label="Speak your vegetable prompt...")
    image_output = gr.Image(label="Generated Result")
    transcript_output = gr.Textbox(label="What Whisper Heard")
    submit_btn = gr.Button("Generate from Voice")

    submit_btn.click(
        fn=audio_to_image,
        inputs=audio_input,
        outputs=[image_output, transcript_output]
    )

stage_2.launch(debug=True)

It looks like you are running Gradio on a hosted Jupyter notebook, which requires `share=True`. Automatically setting `share=True` (you can turn this off by setting `share=False` in `launch()` explicitly).

Colab notebook detected. This cell will run indefinitely so that you can see errors and logs. To turn off, set debug=False in launch().
* Running on public URL: https://cbd16b8be352d33e19.gradio.live

This share link expires in 1 week. For free permanent hosting and GPU upgrades, run `gradio deploy` from the terminal in the working directory to deploy to Hugging Face Spaces (https://huggingface.co/spaces)


Keyboard interruption in main thread... closing server.
Killing tunnel 127.0.0.1:7860 <> https://cbd16b8be352d33e19.gradio.live


In [31]:
def transcribe_voice(audio_path):
    """Helper function to update the text box as soon as they stop talking"""
    if audio_path is None:
        return ""
    with open(audio_path, "rb") as audio_file:
        transcript = client.audio.transcriptions.create(
            model="whisper-1",
            file=audio_file
        )
    return transcript.text

# Stage 3 UI (Fully Integrated)
with gr.Blocks(theme=gr.themes.Soft()) as final_app:
    gr.Markdown("# 🥕 Vegetable Image Architect AI")
    gr.Markdown("Provide instructions using your voice, text, or both!")

    with gr.Row():
        with gr.Column():
            audio_in = gr.Audio(type="filepath", label="Option A: Talk to Microphone")
            text_in = gr.Textbox(label="Option B: Type or Edit Prompt Here", lines=3)
            generate_btn = gr.Button("🚀 Generate Image", variant="primary")

        with gr.Column():
            img_out = gr.Image(label="Generated Output Asset")

    # 1. When audio finishes recording, automatically transcribe it directly INTO the text box
    audio_in.change(fn=transcribe_voice, inputs=audio_in, outputs=text_in)

    # 2. Clicking the generate button reads whatever text is inside the text box
    generate_btn.click(fn=text_to_image, inputs=text_in, outputs=img_out)

final_app.launch(share=True)

/tmp/ipykernel_3000/3177749469.py:13: DeprecationWarning: The 'theme' parameter in the Blocks constructor will be removed in Gradio 6.0. You will need to pass 'theme' to Blocks.launch() instead.
  with gr.Blocks(theme=gr.themes.Soft()) as final_app:


Colab notebook detected. To show errors in colab notebook, set debug=True in launch()
* Running on public URL: https://8d0396189470ae91b0.gradio.live

This share link expires in 1 week. For free permanent hosting and GPU upgrades, run `gradio deploy` from the terminal in the working directory to deploy to Hugging Face Spaces (https://huggingface.co/spaces)
